In [117]:
#file: LHMS_850011035.ipynb
#name: Aditya Verma
#student id: 850011035
#date: 06 november 2026
#description: manage hotel rooms, bookings, bills and files

#this will track whether required modules loaded successfully
#the application must not start if an import fails
modules_ready = False

try:
    #this will provide booking dates and backup timestamps
    #timedelta will calculate checkout dates from booked nights
    from datetime import datetime, timedelta

    #this will represent prices using decimal numbers
    #InvalidOperation will handle invalid price conversions
    from decimal import Decimal, InvalidOperation

    #this will replace saved files after writing succeeds
    from os import replace

    #this will confirm that all required imports succeeded
    modules_ready = True

except ImportError:
    #this will explain a required module import failure
    print("A required module could not be loaded.")
    print("Check the imports before starting the application.")

except SyntaxError:
    #this will handle syntax errors inside imported modules
    #it cannot catch invalid syntax in this cell itself
    print("An imported module contains a syntax error.")
    print("Correct the module before starting the application.")

#this will identify the student's allocation files
STUDENT_ID = "850011035"

#this will name the main allocation text file
FILE_NAME = f"LHMS_{STUDENT_ID}.txt"

#this will provide the backup filename pattern
#the backup method will insert the actual timestamp
BACKUP_TEMPLATE = f"LHMS_{STUDENT_ID}_Backup_{{}}.txt"

#this will confirm successful setup
if modules_ready:
    print("Hotel application setup is ready.")
    print(f"Allocation filename: {FILE_NAME}")

Hotel application setup is ready.
Allocation filename: LHMS_850011035.txt


In [118]:
#this will request a valid whole number
def get_number(message, minimum=1, maximum=None):
    """Return a whole number within the specified limits."""

    #this will repeat the question until input is valid
    while True:
        try:
            #this will convert entered text into an integer
            number = int(input(message))

            #this will reject numbers below the minimum
            if number < minimum:
                print(f"Enter a number of at least {minimum}.")
                continue

            #this will check the maximum when one is provided
            if maximum is not None and number > maximum:
                print(f"Enter a number no greater than {maximum}.")
                continue

            #this will return the accepted number
            return number

        except ValueError:
            #this will handle letters or decimal input
            print("Invalid input. Enter a whole number.")


#this will request nonblank text on one line
def get_text(message):
    """Return nonblank, single-line text that UTF-8 can store."""

    while True:
        #this will read the user's original entry
        text = input(message)

        #this will reject line breaks before removing spaces
        if "\n" in text or "\r" in text:
            print("Enter the details on one line.")
            continue

        #this will remove surrounding spaces
        text = text.strip()

        #this will reject an empty entry
        if text == "":
            print("This field cannot be blank.")
            continue

        try:
            #this will check that the text can be saved
            text.encode("utf-8")

        except UnicodeEncodeError:
            print("This text contains unsupported characters.")
            continue

        #this will return the accepted text
        return text


#this will request a valid nightly price
def get_price(message):
    """Return a positive Decimal price with at most two decimal places."""

    while True:
        try:
            #this will convert entered text into decimal money
            price_text = input(message).strip()
            price = Decimal(price_text)

            #this will reject infinity and undefined values
            if not price.is_finite():
                print("Enter a finite price, such as 150.00.")
                continue

            #this will reject zero and negative prices
            if price <= 0:
                print("The nightly price must be greater than zero.")
                continue

            #this will prevent excessively large billing values
            if price > Decimal("100000.00"):
                print("The maximum nightly price is NZD $100,000.00.")
                continue

            #this will check the number of decimal places
            if price.as_tuple().exponent < -2:
                print("Use no more than two decimal places.")
                continue

            #this will return the accepted decimal price
            return price

        except InvalidOperation:
            #this will handle text that cannot become a price
            print("Invalid price. Enter a number such as 150.00.")

In [119]:
#this will group the hotel's records and operations
class HotelManager:
    """Manage hotel rooms, bookings, billing and allocation files."""

    #this will prepare a new hotel object
    def __init__(self):
        """Create empty records and assign the file settings."""

        #this will store rooms using unique room numbers
        #each record will contain type, features and nightly rate
        self.rooms = {}

        #this will store active bookings using room numbers
        #a room is occupied when its number appears here
        self.bookings = {}

        #this will provide the three room type choices
        self.room_types = ["Single", "Double", "Suite"]

        #this will use the allocation filename from cell one
        #the save operation will create the actual file later
        self.file_name = FILE_NAME

        #this will use the backup pattern from cell one
        #the backup operation will insert the date and time
        self.backup_template = BACKUP_TEMPLATE
            #this will collect and save a new room
    def add_room(self):
        """Validate room details before adding a new room."""

        #this will request a positive whole room number
        room_number = get_number("Enter room number: ")

        #this will prevent replacing an existing room
        if room_number in self.rooms:
            print("Room number already exists.")
            return

        #this will display the available room types
        print("1. Single")
        print("2. Double")
        print("3. Suite")

        #this will accept only the displayed choices
        type_choice = get_number("Choose room type (1-3): ", 1, 3)

        try:
            #this will convert the choice into a list position
            #choice one uses position zero because lists start there
            room_type = self.room_types[type_choice - 1]

        except IndexError:
            #this will handle a missing room type list entry
            #input limits alone cannot detect an incomplete list
            print("The selected room type is missing from the list.")
            print("Check the room type settings.")
            return

        #this will collect nonblank room features
        features = get_text("Enter room features: ")

        #this will collect a validated nightly price
        rate = get_price("Enter nightly rate (NZD): ")

        #this will save only the complete validated room
        #interrupted input will not leave a partial room record
        self.rooms[room_number] = {
            "type": room_type,
            "features": features,
            "rate": rate
        }

        print(f"Room {room_number} added successfully.")

    #this will remove an unoccupied room
    def delete_room(self):
        """Delete an existing room only when it has no booking."""

        #this will request the room number to remove
        room_number = get_number("Enter room number to delete: ")

        #this will check whether the room exists
        if room_number not in self.rooms:
            print("Room number not found.")
            return

        #this will protect rooms with active bookings
        if room_number in self.bookings:
            print("Cannot delete an occupied room.")
            return

        #this will remove the available room record
        del self.rooms[room_number]
        print(f"Room {room_number} deleted successfully.")

    #this will display room details before booking
    def display_rooms(self):
        """Show each room's features, nightly rate and availability."""

        #this will explain when no rooms exist
        if not self.rooms:
            print("No rooms added yet.")
            return

        #this will display rooms in room number order
        for room_number in sorted(self.rooms):
            room = self.rooms[room_number]

            #this will determine availability from current bookings
            if room_number in self.bookings:
                status = "Occupied"
            else:
                status = "Available"

            #this will display the complete room information
            print(f"\nRoom number: {room_number}")
            print(f"Room type: {room['type']}")
            print(f"Features: {room['features']}")
            print(f"Nightly rate: NZD ${room['rate']:.2f}")
            print(f"Status: {status}")
        #this will allocate an available room to a customer
    def allocate_room(self):
        """Validate booking details before recording an allocation."""

        #this will prevent booking when no rooms exist
        if not self.rooms:
            print("No rooms exist. Add a room first.")
            return

        #this will request the room to allocate
        room_number = get_number("Enter room number to allocate: ")

        #this will check that the room exists
        if room_number not in self.rooms:
            print("Room number not found.")
            return

        #this will prevent allocating an occupied room again
        if room_number in self.bookings:
            print("Room is already occupied.")
            return

        #this will collect validated customer details
        customer_number = get_text("Enter customer number: ")
        customer_name = get_text("Enter customer name: ")

        #this will repeat date entry until valid
        while True:
            try:
                #this will read and convert the check-in date
                date_text = get_text("Enter check-in date (YYYY-MM-DD): ")
                check_in = datetime.strptime(date_text, "%Y-%m-%d").date()

                #this will enforce the exact displayed date format
                if date_text != check_in.isoformat():
                    raise ValueError("Use YYYY-MM-DD.")

                #this will request a positive number of nights
                nights = get_number("Enter number of nights: ")

                #this will calculate the checkout date
                #dates outside the supported range cause OverflowError
                check_out = check_in + timedelta(days=nights)

                #this will finish the date validation loop
                break

            except ValueError:
                #this will explain an invalid date or format
                print("Enter a valid calendar date using YYYY-MM-DD.")

            except OverflowError:
                #this will handle an unsupported checkout date
                print("The stay exceeds the supported date range.")
                print("Enter the check-in date and nights again.")

        #this will preserve the agreed nightly price
        #later room price changes will not alter this booking
        agreed_rate = self.rooms[room_number]["rate"]

        #this will save only the complete validated booking
        #the booking record also marks the room as occupied
        self.bookings[room_number] = {
            "customer_number": customer_number,
            "customer_name": customer_name,
            "check_in": check_in,
            "check_out": check_out,
            "nights": nights,
            "rate": agreed_rate
        }

        print(f"Room {room_number} allocated to {customer_name}.")
        print(f"Check-in date: {check_in}")
        print(f"Checkout date: {check_out}")

    #this will display active bookings and room occupancy
    def display_allocations(self):
        """Show booking details and the percentage of occupied rooms."""

        #this will explain when no bookings exist
        if not self.bookings:
            print("No rooms are currently allocated.")

        #this will display bookings in room number order
        for room_number in sorted(self.bookings):
            booking = self.bookings[room_number]

            #this will display the saved customer and stay details
            print(f"\nRoom number: {room_number}")
            print(f"Customer number: {booking['customer_number']}")
            print(f"Customer name: {booking['customer_name']}")
            print(f"Check-in date: {booking['check_in']}")
            print(f"Checkout date: {booking['check_out']}")
            print(f"Nights: {booking['nights']}")
            print(f"Agreed nightly rate: NZD ${booking['rate']:.2f}")

        try:
            #this will calculate the percentage of occupied rooms
            occupancy = len(self.bookings) / len(self.rooms) * 100
            print(f"\nOccupancy: {occupancy:.1f}%")

        except ZeroDivisionError:
            #this will handle occupancy when no rooms exist
            print("Occupancy unavailable: no rooms have been added.")
    #this will calculate the accommodation charge
    def calculate_bill(self, rate, nights):
        """Return the agreed nightly rate multiplied by nights."""

        #this will check the data types before calculating
        #type checks also prevent treating true as one night
        if not isinstance(rate, Decimal) or type(nights) is not int:
            raise TypeError("Billing requires a Decimal rate and integer nights.")

        #this will reject undefined or infinite prices
        if not rate.is_finite():
            raise ValueError("The nightly rate must be finite.")

        #this will check the allowed price range
        if rate <= 0 or rate > Decimal("100000.00"):
            raise ValueError("The nightly rate must be above zero and at most 100000.")

        #this will prevent billing for zero or negative nights
        if nights <= 0:
            raise ValueError("Nights must be greater than zero.")

        #this will check stored prices for excessive decimal places
        if rate.as_tuple().exponent < -2:
            raise ValueError("The nightly rate must have at most two decimal places.")

        #this will calculate the total using the agreed rate
        return rate * nights

    #this will display the bill before releasing the room
    def billing_deallocation(self):
        """Generate a bill and release the room after confirmation."""

        #this will request a positive whole room number
        room_number = get_number("Enter room number for checkout: ")

        #this will check that the room has an active booking
        if room_number not in self.bookings:
            print("This room has no current booking.")
            return

        try:
            #this will retrieve the current booking
            booking = self.bookings[room_number]

            #this will calculate before removing any booking
            #failed billing must leave the room occupied
            total = self.calculate_bill(
                booking["rate"],
                booking["nights"]
            )

            #this will display an itemised accommodation bill
            print("\n=== LANGHAM HOTEL BILL ===")
            print(f"Room number: {room_number}")
            print(f"Customer number: {booking['customer_number']}")
            print(f"Customer name: {booking['customer_name']}")
            print(f"Check-in date: {booking['check_in']}")
            print(f"Checkout date: {booking['check_out']}")
            print(f"Nights: {booking['nights']}")
            print(f"Nightly rate: NZD ${booking['rate']:.2f}")
            print(f"Total bill: NZD ${total:.2f}")

            #this will repeat until confirmation is understood
            while True:
                confirm = get_text("Confirm checkout? (yes/no): ").lower()

                if confirm == "yes":
                    #this will remove only the completed booking
                    #the room remains available for another customer
                    del self.bookings[room_number]
                    print(f"Checkout complete. Room {room_number} is available.")
                    break

                elif confirm == "no":
                    #this will preserve the active booking
                    print("Checkout cancelled. The booking remains active.")
                    break

                else:
                    print("Please enter yes or no.")

        except TypeError as error:
            #this will report incorrect billing data types
            print(f"Billing data error: {error}")
            print("Correct the data before checking out.")

        except ValueError as error:
            #this will report invalid stored billing values
            print(f"Billing value error: {error}")
            print("Correct the data before checking out.")
        #this will save a snapshot of current bookings
    def save_allocations(self):
        """Write current allocations and safely replace the saved file."""

        #this will prepare the text before opening any file
        content = "LANGHAM HOTEL ROOM ALLOCATIONS\n"
        content += f"Saved: {datetime.now():%Y-%m-%d %H:%M:%S}\n\n"

        #this will explain when no bookings are active
        if not self.bookings:
            content += "No rooms are currently allocated.\n"

        #this will include each booking in room number order
        for room_number in sorted(self.bookings):
            booking = self.bookings[room_number]

            content += f"Room number: {room_number}\n"
            content += f"Customer number: {booking['customer_number']}\n"
            content += f"Customer name: {booking['customer_name']}\n"
            content += f"Check-in date: {booking['check_in']}\n"
            content += f"Checkout date: {booking['check_out']}\n"
            content += f"Nights: {booking['nights']}\n"
            content += f"Agreed nightly rate: NZD ${booking['rate']:.2f}\n"
            content += "----------------------------------------\n"

        #this will keep new writing separate from previous records
        temporary_name = self.file_name + ".tmp"

        try:
            #this will create or overwrite the temporary file
            #with closes it even if writing raises an error
            with open(temporary_name, "w", encoding="utf-8") as file:
                file.write(content)

            #this will replace the allocation file after writing succeeds
            #it also creates the final file if none exists
            replace(temporary_name, self.file_name)

            print(f"Allocations saved to {self.file_name}")

        except UnicodeEncodeError:
            #this will handle text that cannot be encoded
            print("Some booking text cannot be saved as UTF-8.")
            print("The previous allocation file was not replaced.")

        except IOError as error:
            #this will handle writing and replacement failures
            print(f"Could not save allocations: {error}")

    #this will display records from the saved allocation file
    def display_file(self):
        """Read saved text without restoring bookings into memory."""

        try:
            #this will open and read the allocation file
            #with closes the file after reading
            with open(self.file_name, "r", encoding="utf-8") as file:
                content = file.read()

            #this will distinguish an empty file from saved records
            if content == "":
                print("The allocation file is empty.")
            else:
                print("\n=== SAVED ROOM ALLOCATIONS ===")
                print(content)

        except FileNotFoundError:
            #this will explain when no saved file exists
            #this specific handler comes before the broader IOError
            print("Allocation file not found. Use option 7 first.")

        except UnicodeDecodeError:
            #this will handle files containing invalid UTF-8 text
            print("The allocation file cannot be read as UTF-8 text.")
            print("Check the file encoding before trying again.")

        except IOError as error:
            #this will handle other file reading failures
            print(f"Could not read allocations: {error}")  
        #this will protect saved records before clearing them
    def backup_and_clear_file(self):
        """Append records to a verified backup before clearing the original."""

        try:
            #this will read the saved allocation contents
            #with closes the original after reading
            with open(self.file_name, "r", encoding="utf-8") as file:
                content = file.read()

            #this will avoid creating an empty backup
            if content == "":
                print("The allocation file is already empty.")
                return

            #this will create the backup date and time
            #microseconds help distinguish closely timed backups
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
            backup_name = self.backup_template.format(timestamp)

            #this will open the backup for reading and appending
            #a+ creates the file if it does not exist
            with open(backup_name, "a+", encoding="utf-8") as backup:

                #this will read any previous backup contents
                backup.seek(0)
                previous_content = backup.read()

                #this will move to the end before appending
                #two means the position is measured from the end
                backup.seek(0, 2)
                backup.write(content)

            #this will check the backup after writing and closing
            with open(backup_name, "r", encoding="utf-8") as backup:
                saved_content = backup.read()

            #this will verify the original and appended contents
            #verification failure must not clear the original
            if saved_content != previous_content + content:
                raise IOError("Backup verification failed. Original not cleared.")

            #this will clear the original after verification succeeds
            #the file remains present with no contents
            with open(self.file_name, "w", encoding="utf-8") as file:
                file.write("")

            #this will confirm the completed operations
            print(f"Backup created and verified: {backup_name}")
            print(f"Allocation file cleared: {self.file_name}")
            print("Current bookings remain active in memory.")

        except FileNotFoundError:
            #this will handle a missing required file
            print("A required file was not found. Check option 7 first.")

        except UnicodeDecodeError:
            #this will stop when saved text cannot be decoded
            #the original has not been cleared at this stage
            print("A file cannot be read as UTF-8 text.")
            print("Backup stopped. The original file was not cleared.")

        except IOError as error:
            #this will report reading, writing or verification failures
            print(f"Backup and clear operation incomplete: {error}")

In [120]:
#this will connect menu choices to hotel operations
def menu(hotel):
    """Repeat the menu until exit or a handled stopping condition."""

    #this will show the menu again after each operation
    while True:
        print("\n=== LANGHAM HOTEL MANAGEMENT SYSTEM ===")
        print("1. Add Room")
        print("2. Delete Room")
        print("3. Display Room Details")
        print("4. Allocate Room")
        print("5. Display Room Allocation Details")
        print("6. Billing and Deallocation")
        print("7. Save Room Allocations to File")
        print("8. Display Room Allocations from File")
        print("9. Backup and Clear Allocation File")
        print("0. Exit Application")
        print("======================================")

        try:
            #this will accept only choices zero to nine
            #the helper repeats the prompt after invalid input
            choice = get_number("Enter your choice (0-9): ", 0, 9)

            #this will call the selected hotel method
            if choice == 1:
                hotel.add_room()

            elif choice == 2:
                hotel.delete_room()

            elif choice == 3:
                hotel.display_rooms()

            elif choice == 4:
                hotel.allocate_room()

            elif choice == 5:
                hotel.display_allocations()

            elif choice == 6:
                hotel.billing_deallocation()

            elif choice == 7:
                hotel.save_allocations()

            elif choice == 8:
                hotel.display_file()

            elif choice == 9:
                hotel.backup_and_clear_file()

            elif choice == 0:
                #this will leave the menu without deleting files
                print("Hotel application closed.")
                break

        except EOFError:
            #this will handle input ending without a response
            #it also covers input inside the called hotel methods
            print("\nInput ended. The hotel menu has stopped.")
            print("Unsaved changes have not been saved automatically.")
            break

        except NameError as error:
            #this will report an undefined function or variable
            #execution stops so the programming fault can be corrected
            print(f"Program name error: {error}")
            print("Check the code and run notebook cells in order.")
            break

In [121]:
#this will check setup before starting the hotel application
try:
    if modules_ready:
        #this will create a new hotel with empty records
        hotel = HotelManager()

        #this will explain the limits of saved file records
        print("New session. Rooms and bookings start empty.")
        print("Saved allocations can be viewed using option 8.")
        print("Saved records are not automatically restored.")

        #this will open the menu for this hotel object
        menu(hotel)

    else:
        #this will stop startup when required imports failed
        print("Required imports failed.")
        print("Correct the imports and run Cell 1 again.")

except NameError as error:
    #this will explain missing notebook definitions
    #this can happen when cells run out of order
    print(f"Startup name error: {error}")
    print("Run Cells 1 to 4 in order, then run Cell 5.")

New session. Rooms and bookings start empty.
Saved allocations can be viewed using option 8.
Saved records are not automatically restored.

=== LANGHAM HOTEL MANAGEMENT SYSTEM ===
1. Add Room
2. Delete Room
3. Display Room Details
4. Allocate Room
5. Display Room Allocation Details
6. Billing and Deallocation
7. Save Room Allocations to File
8. Display Room Allocations from File
9. Backup and Clear Allocation File
0. Exit Application


Enter your choice (0-9):  1
Enter room number:  101


1. Single
2. Double
3. Suite


Choose room type (1-3):  1
Enter room features:  wifi and single bed
Enter nightly rate (NZD):  150


Room 101 added successfully.

=== LANGHAM HOTEL MANAGEMENT SYSTEM ===
1. Add Room
2. Delete Room
3. Display Room Details
4. Allocate Room
5. Display Room Allocation Details
6. Billing and Deallocation
7. Save Room Allocations to File
8. Display Room Allocations from File
9. Backup and Clear Allocation File
0. Exit Application


Enter your choice (0-9):  1
Enter room number:  102


... (27 outputs left out)

Enter your choice (0-9):  3



Room number: 101
Room type: Single
Features: wifi and single bed
Nightly rate: NZD $150.00
Status: Available

Room number: 102
Room type: Double
Features: wifi and double bed
Nightly rate: NZD $220.00
Status: Available

=== LANGHAM HOTEL MANAGEMENT SYSTEM ===
1. Add Room
2. Delete Room
3. Display Room Details
4. Allocate Room
5. Display Room Allocation Details
6. Billing and Deallocation
7. Save Room Allocations to File
8. Display Room Allocations from File
9. Backup and Clear Allocation File
0. Exit Application


Enter your choice (0-9):  0


Hotel application closed.


In [94]:
#controlled fault: the function definition is missing a colon
def show_room_details(room_number, room_type)
    print(f"Room number: {room_number}")
    print(f"Room type: {room_type}")

show_room_details(101, "Single")

SyntaxError: expected ':' (2406123835.py, line 2)

In [95]:
#this will define the function with the required colon
def show_room_details(room_number, room_type):
    """Display the room number and type."""

    #this will display the supplied room details
    print(f"Room number: {room_number}")
    print(f"Room type: {room_type}")

#this will retest the function with the same sample data
show_room_details(101, "Single")

Room number: 101
Room type: Single


In [96]:
#controlled fault: room input is converted without error handling
test_room_number = int(input("Enter room number: "))

#this line will not run when conversion fails
print(f"Room number accepted: {test_room_number}")

Enter room number:  11


Room number accepted: 11


In [97]:
#this will use the application's existing validation helper
#the helper catches ValueError and repeats the question
test_room_number = get_number("Enter room number: ")

#this will display the accepted positive whole number
print(f"Room number accepted: {test_room_number}")

Enter room number:  11


Room number accepted: 11


In [98]:
#this will create a separate hotel for the example
naming_test_hotel = HotelManager()

#controlled fault: the variable name is misspelled
naming_test_hotl.display_rooms()

NameError: name 'naming_test_hotl' is not defined

In [99]:
#this will create a separate hotel for the corrected example
naming_test_hotel = HotelManager()

#this will use the correctly spelled variable name
naming_test_hotel.display_rooms()

No rooms added yet.


In [100]:
#this will create a separate hotel for billing tests
billing_test_hotel = HotelManager()

#controlled fault: the nightly rate is stored as text
test_rate = "150.00"
test_nights = 3

#this will call the application's actual bill calculator
test_total = billing_test_hotel.calculate_bill(test_rate, test_nights)
print(f"Total bill: NZD ${test_total:.2f}")

TypeError: Billing requires a Decimal rate and integer nights.

In [101]:
#this will create a separate hotel for the corrected test
billing_test_hotel = HotelManager()

#this will store the nightly rate as decimal money
test_rate = Decimal("150.00")
test_nights = 3

#this will retest the same calculation with correct types
test_total = billing_test_hotel.calculate_bill(test_rate, test_nights)
print(f"Total bill: NZD ${test_total:.2f}")

Total bill: NZD $450.00


In [102]:
#this will create a separate hotel for the example
index_test_hotel = HotelManager()

#controlled fault: the choice is used directly as an index
test_type_choice = 3
selected_type = index_test_hotel.room_types[test_type_choice]

print(f"Selected room type: {selected_type}")

IndexError: list index out of range

In [103]:
#this will create a separate hotel for the corrected example
index_test_hotel = HotelManager()

#this will represent the user's third menu choice
test_type_choice = 3

#this will convert the choice into the correct list position
#subtracting one changes choice three into position two
selected_type = index_test_hotel.room_types[test_type_choice - 1]

print(f"Selected room type: {selected_type}")

Selected room type: Suite


In [104]:
#this will provide a temporary folder for the example
from tempfile import TemporaryDirectory

with TemporaryDirectory() as folder:
    #this will name a file that has not been created
    missing_file = folder + "/LHMS_85001138.txt"

    #controlled fault: reading before saving without error handling
    with open(missing_file, "r", encoding="utf-8") as file:
        print(file.read())

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\av563\\AppData\\Local\\Temp\\tmp263mu1qr/LHMS_85001138.txt'

In [105]:
#this will provide a separate temporary test folder
from tempfile import TemporaryDirectory

with TemporaryDirectory() as folder:
    #this will create a hotel without changing main records
    file_test_hotel = HotelManager()
    file_test_hotel.file_name = folder + "/LHMS_85001138.txt"

    #this will use the application's missing-file handler
    print("\n=== Read before saving ===")
    file_test_hotel.display_file()

    #this will create the allocation file before reading again
    print("\n=== Save the allocation snapshot ===")
    file_test_hotel.save_allocations()

    #this will read the file after successful saving
    print("\n=== Read after saving ===")
    file_test_hotel.display_file()


=== Read before saving ===
Allocation file not found. Use option 7 first.

=== Save the allocation snapshot ===
Allocations saved to C:\Users\av563\AppData\Local\Temp\tmpuyodzraq/LHMS_85001138.txt

=== Read after saving ===

=== SAVED ROOM ALLOCATIONS ===
LANGHAM HOTEL ROOM ALLOCATIONS
Saved: 2026-11-06 21:52:40

No rooms are currently allocated.



In [106]:
#this will test a hotel-related syntax error
print("\n=== X1: SyntaxError handling ===")

try:
    #this example deliberately omits the function's colon
    hotel_source = "def display_rooms(self)\n    pass\n"

    #this will check syntax without executing the example
    compile(hotel_source, "hotel_syntax_test", "exec")

except SyntaxError:
    #this will explain the fault without a red traceback
    print("Hotel function syntax error: a colon is missing.")


#this will test an undefined hotel class name
print("\n=== X2: NameError handling ===")

try:
    #this deliberately uses a class name that does not exist
    test_object = HotelManager_NameFault()

except NameError:
    #this will explain the incorrect program name
    print("Hotel class name is undefined. Check its spelling.")


#this will test an incorrect date-module import
print("\n=== X3: ImportError handling ===")

try:
    #this deliberately requests a nonexistent module component
    from datetime import hotel_missing_component

except ImportError:
    #this will explain the failed import
    print("Hotel date component could not be imported.")


=== X1: SyntaxError handling ===
Hotel function syntax error: a colon is missing.

=== X2: NameError handling ===
Hotel class name is undefined. Check its spelling.

=== X3: ImportError handling ===
Hotel date component could not be imported.


In [108]:
#this will create a separate hotel for controlled tests
handler_test_hotel = HotelManager()

#this will deliberately make the room type list incomplete
handler_test_hotel.room_types = ["Single"]

#this will test the actual IndexError handler
print("\n=== X7: IndexError handling ===")
print("Enter room number 101, then room type 3.")
handler_test_hotel.add_room()

#this will restore the correct room type settings
handler_test_hotel.room_types = ["Single", "Double", "Suite"]

#this will add the approved sample room
handler_test_hotel.rooms[101] = {
    "type": "Single",
    "features": "Wi-Fi, single bed",
    "rate": Decimal("150.00")
}

#this will deliberately store a booking rate as text
handler_test_hotel.bookings[101] = {
    "customer_number": "C001",
    "customer_name": "Alex Taylor",
    "check_in": "2026-11-06",
    "check_out": "2026-11-09",
    "nights": 3,
    "rate": "150.00"
}

#this will test the actual checkout TypeError handler
print("\n=== X8: TypeError handling ===")
print("Enter room number 101.")
handler_test_hotel.billing_deallocation()

#this will check that failed billing preserved the booking
print("Booking remains active:", 101 in handler_test_hotel.bookings)

#this will restore the correct billing data type
handler_test_hotel.bookings[101]["rate"] = Decimal("150.00")


=== X7: IndexError handling ===
Enter room number 101, then room type 3.


Enter room number:  1101


1. Single
2. Double
3. Suite


Choose room type (1-3):  3


The selected room type is missing from the list.
Check the room type settings.

=== X8: TypeError handling ===
Enter room number 101.


Enter room number for checkout:  101


Billing data error: Billing requires a Decimal rate and integer nights.
Correct the data before checking out.
Booking remains active: True


In [109]:
#this will isolate test files from actual allocation records
from tempfile import TemporaryDirectory

#this will create a directory for the controlled save failure
from os import mkdir

with TemporaryDirectory() as folder:
    #this will create a separate hotel for file tests
    file_handler_test = HotelManager()

    #this will point to an allocation file that does not exist
    file_handler_test.file_name = folder + "/missing_allocations.txt"

    #this will test the actual missing-file handler
    print("\n=== X9: FileNotFoundError handling ===")
    file_handler_test.display_file()

    #this will create a directory at the intended file destination
    #a text file cannot replace this directory
    blocked_name = folder + "/blocked_allocations.txt"
    mkdir(blocked_name)
    file_handler_test.file_name = blocked_name

    #this will test the actual save failure handler
    print("\n=== X10: IOError handling ===")
    file_handler_test.save_allocations()

    #this will restore a valid allocation filename
    file_handler_test.file_name = folder + "/valid_allocations.txt"

    #this will check saving and reading after correction
    print("\n=== Valid destination retest ===")
    file_handler_test.save_allocations()
    file_handler_test.display_file()


=== X9: FileNotFoundError handling ===
Allocation file not found. Use option 7 first.

=== X10: IOError handling ===
Could not save allocations: [WinError 5] Access is denied: 'C:\\Users\\av563\\AppData\\Local\\Temp\\tmp1t8hgnqw/blocked_allocations.txt.tmp' -> 'C:\\Users\\av563\\AppData\\Local\\Temp\\tmp1t8hgnqw/blocked_allocations.txt'

=== Valid destination retest ===
Allocations saved to C:\Users\av563\AppData\Local\Temp\tmp1t8hgnqw/valid_allocations.txt

=== SAVED ROOM ALLOCATIONS ===
LANGHAM HOTEL ROOM ALLOCATIONS
Saved: 2026-11-06 21:53:30

No rooms are currently allocated.



In [111]:
#this will read the saved notebook's structure
import json

#this will run a separate Python process
import subprocess
import sys

#this will keep the test script temporary
from tempfile import TemporaryDirectory

#this will load the saved notebook
with open("LHMS_850011035.ipynb", "r", encoding="utf-8") as file:
    notebook = json.load(file)

#this will collect code cells in notebook order
code_cells = []

for cell in notebook["cells"]:
    if cell["cell_type"] == "code":
        code_cells.append("".join(cell["source"]))

#this will copy only setup, helpers, class and menu
#the intentional error demonstrations are excluded
application_source = "\n\n".join(code_cells[:4])

with TemporaryDirectory() as folder:
    #this will create a temporary application script
    script_name = folder + "/hotel_eof_test.py"

    with open(script_name, "w", encoding="utf-8") as file:
        file.write(application_source)
        file.write("\n\nmenu(HotelManager())\n")

    #this will supply no input to the application
    #input() will encounter EOF instead of a response
    result = subprocess.run(
        [sys.executable, script_name],
        input="",
        capture_output=True,
        text=True
    )

    #this will display the application's captured output
    print(result.stdout)

    #this will report unexpected test failures
    if result.returncode != 0:
        print("Unexpected test failure:")
        print(result.stderr)

Hotel system setup ready
Allocation file- LHMS_850011035.txt

LANGHAM HOTEL MANAGEMENT SYSTEM
1. Add room
2. Delete room
3. Display room details
4. Allocate room
5. Display room allocation details
6. Billing and deallocation
7. Save Room allocations to file
8. Display Room allocations from file
9. Backup and clear allocation file
0. Exit
Enter your choice (0-9)- 
Input ended. Hotel menu stopped.
Unsaved records have not been written to file

LANGHAM HOTEL MANAGEMENT SYSTEM
1. Add room
2. Delete room
3. Display room details
4. Allocate room
5. Display room allocation details
6. Billing and deallocation
7. Save Room allocations to file
8. Display Room allocations from file
9. Backup and clear allocation file
0. Exit
Enter your choice (0-9)- 
Input ended. Hotel menu stopped.
Unsaved records have not been written to file

